# PubMedBERT Fine-tuning Evaluation

Evaluating the fine-tuned PubMedBERT model and comparing against the TF-IDF baselines.

In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')

In [2]:
with open('../outputs/eval_results.json') as f:
    results = json.load(f)

print(f'Overall accuracy: {results["accuracy"]:.4f}')
print(f'Macro F1:         {results["macro_f1"]:.4f}')
print(f'Weighted F1:      {results["weighted_f1"]:.4f}')

Overall accuracy: 0.8913
Macro F1:         0.8847
Weighted F1:      0.8911


## Per-class Metrics

In [3]:
print(f'{"":15s} {"Precision":>10s} {"Recall":>8s} {"F1":>8s} {"Support":>9s}')
for label, metrics in results['per_class'].items():
    print(f'{label:15s} {metrics["precision"]:10.4f} {metrics["recall"]:8.4f} {metrics["f1"]:8.4f} {metrics["support"]:9.0f}')

                Precision   Recall       F1   Support
BACKGROUND         0.8724   0.8581   0.8652      4378
OBJECTIVE          0.8891   0.8534   0.8709      3524
METHODS            0.9012   0.9234   0.9122      8892
RESULTS            0.9087   0.9143   0.9115      9741
CONCLUSIONS        0.8643   0.8421   0.8531      3600


In [4]:
labels = list(results['per_class'].keys())
precision = [results['per_class'][l]['precision'] for l in labels]
recall = [results['per_class'][l]['recall'] for l in labels]
f1 = [results['per_class'][l]['f1'] for l in labels]

x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, precision, width, label='Precision', color='#2196F3')
ax.bar(x, recall, width, label='Recall', color='#4CAF50')
ax.bar(x + width, f1, width, label='F1', color='#FF9800')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Score')
ax.set_title('Per-class Metrics - PubMedBERT')
ax.legend()
ax.set_ylim(0.8, 0.95)
plt.tight_layout()
plt.show()

<Figure size 1000x500 with 1 Axes>

METHODS and RESULTS consistently highest across all metrics. CONCLUSIONS is still the weakest class but jumped from 0.77 to 0.85 F1 compared to baseline.

## Confusion Matrix

In [5]:
# Approximate confusion matrix from eval run
cm = np.array([
    [3756,  187,  148,  102,  185],
    [ 172, 3007,   98,   87,  160],
    [  89,   65, 8211,  412,  115],
    [  71,   52,  341, 8907,  370],
    [ 143,   72,  115,  238, 3032],
])

labels = ['BACKGROUND', 'OBJECTIVE', 'METHODS', 'RESULTS', 'CONCLUSIONS']

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

<Figure size 800x600 with 2 Axes>

Main confusion patterns:
- RESULTS <-> CONCLUSIONS (makes sense, both discuss findings)
- BACKGROUND <-> OBJECTIVE (both are intro-type sentences)
- METHODS <-> RESULTS (sometimes hard to distinguish procedural from reporting)

## Comparison: Baseline vs PubMedBERT

In [6]:
print('Model Comparison:
')
print(f'{"":20s} {"Accuracy":>10s} {"Macro F1":>10s} {"Weighted F1":>13s}')
print(f'{"TF-IDF + LogReg":20s} {0.8217:10.4f} {0.8032:10.4f} {0.8208:13.4f}')
print(f'{"TF-IDF + SVM":20s} {0.8284:10.4f} {0.8106:10.4f} {0.8274:13.4f}')
print(f'{"PubMedBERT":20s} {results["accuracy"]:10.4f} {results["macro_f1"]:10.4f} {results["weighted_f1"]:13.4f}')
print(f'
Improvement over best baseline:')
print(f'  Accuracy:    +{(results["accuracy"] - 0.8284)*100:.1f} pp')
print(f'  Macro F1:    +{(results["macro_f1"] - 0.8106)*100:.1f} pp')

Model Comparison:

                    Accuracy   Macro F1   Weighted F1
TF-IDF + LogReg       0.8217     0.8032        0.8208
TF-IDF + SVM          0.8284     0.8106        0.8274
PubMedBERT            0.8913     0.8847        0.8911

Improvement over best baseline:
  Accuracy:    +6.3 pp
  Macro F1:    +7.4 pp


In [7]:
# Per-class F1 comparison
baseline_f1 = [0.7838, 0.7838, 0.8623, 0.8537, 0.7692]  # SVM
bert_f1 = [results['per_class'][l]['f1'] for l in labels]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, baseline_f1, width, label='TF-IDF + SVM', color='#9E9E9E')
ax.bar(x + width/2, bert_f1, width, label='PubMedBERT', color='#2196F3')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('F1 Score')
ax.set_title('Per-class F1: Baseline vs PubMedBERT')
ax.legend()
ax.set_ylim(0.7, 0.95)

# Add improvement annotations
for i in range(len(labels)):
    diff = bert_f1[i] - baseline_f1[i]
    ax.annotate(f'+{diff:.2f}', xy=(x[i] + width/2, bert_f1[i]),
                ha='center', va='bottom', fontsize=8, color='green')

plt.tight_layout()
plt.show()

<Figure size 1000x500 with 1 Axes>

PubMedBERT gives a solid ~7 point improvement across the board. The biggest gains are on OBJECTIVE and CONCLUSIONS, which are the classes where contextual understanding matters most (TF-IDF can't capture that).

89% accuracy on 5-class classification of medical text is a good result. Domain-specific pretraining (PubMedBERT vs generic BERT) likely helps since the vocabulary is medical.